# US Accidents — Loading, Data Preparation & EDA

**Ironhack DSML Final Project — Notebook 1 of 2**

**Goal:** Predict whether a US traffic accident will have a **severe impact on traffic flow** (Severity 3 or 4) based on conditions known at the *start* of the incident.

> **Important:** "Severity" in this dataset measures *impact on traffic / delay duration*, NOT injury severity.

**This notebook covers:**
1. Fast loading (CSV → Parquet cache: ~1-hour load → ~5-second load)
2. **Data Preparation** — missing values, outliers, type casting, feature selection
3. Feature engineering (time + weather grouping)
4. Exploratory Data Analysis with statistical tests
5. Save cleaned data for the modeling notebook

**The hour-long load problem:** `pd.read_csv()` with default settings takes ~1 hour on this 3 GB / 7.7 M-row file because:
- C engine + dtype inference (single-threaded, two passes per column)
- Bloated default dtypes (`object`, `float64`)

**The fix:** Use `engine='pyarrow'` (5–10x faster) and cache to Parquet.

## 0. One-time install
Run once, then comment out.

In [ ]:
!pip install pyarrow scipy --quiet

## 1. Imports

In [ ]:
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 100
pd.set_option("display.max_columns", 60)

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)
FIG_DIR = Path("reports/figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)

CSV_PATH = DATA_DIR / "US_Accidents_March23.csv"
PARQUET_PATH = DATA_DIR / "US_Accidents_March23.parquet"
CLEAN_PARQUET = DATA_DIR / "us_accidents_clean.parquet"

print(f"CSV exists:     {CSV_PATH.exists()}")
print(f"Parquet exists: {PARQUET_PATH.exists()}")

## 2. Fast loading

`load_accidents()` auto-detects: if the Parquet cache exists, use it. Otherwise build it from CSV.

In [ ]:
def load_accidents(csv_path=CSV_PATH, parquet_path=PARQUET_PATH, force_rebuild=False):
    """Load the US Accidents dataset, building a parquet cache on first run."""
    if parquet_path.exists() and not force_rebuild:
        print(f"Loading from parquet cache: {parquet_path}")
        t0 = time.time()
        df = pd.read_parquet(parquet_path)
        print(f"  Loaded in {time.time()-t0:.1f}s — shape: {df.shape}")
        return df

    if not csv_path.exists():
        raise FileNotFoundError(
            f"Couldn't find {csv_path}. Download from "
            "https://www.kaggle.com/datasets/sobhanmoosavi/us-accidents"
        )
    print(f"First-time load from CSV: {csv_path}")
    print("  Using pyarrow engine (much faster than default)...")
    t0 = time.time()
    df = pd.read_csv(csv_path, engine="pyarrow")
    print(f"  Parsed CSV in {time.time()-t0:.1f}s — shape: {df.shape}")
    print(f"  Saving parquet cache to {parquet_path}...")
    t0 = time.time()
    df.to_parquet(parquet_path, compression="snappy", index=False)
    print(f"  Saved in {time.time()-t0:.1f}s")
    return df


df = load_accidents()
print(f"\nMemory: {df.memory_usage(deep=True).sum() / 1e9:.2f} GB")

## 3. Initial inspection

In [ ]:
df.head()

In [ ]:
print(f"Shape: {df.shape}\n")
print("Dtypes:")
print(df.dtypes.value_counts())

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# DATA PREPARATION
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## 4. Data Preparation

Six sub-steps, in order:

1. **Feature selection** — drop target-leakage columns and unused columns (informed selection)
2. **Missing values audit** — quantify what's missing
3. **High-missingness drop** — drop features that are more than 50% missing
4. **Outlier detection & handling** — IQR-based, with clipping (not deletion)
5. **Imputation** — median for numerics, "Unknown" for categoricals
6. **Type casting** — downcast numerics to save memory

### 4.1 Feature selection (informed, by domain logic)

We drop two categories:

**Target leakage** — features describing what happened *after* the accident was recorded. Using these would let the model cheat:

| Column | Why it's leakage |
|---|---|
| `Distance(mi)` | Length of road affected — basically the target restated. |
| `End_Time`, `End_Lat`, `End_Lng` | Only known once the accident has been cleared. |
| `Description` | Free-text written after the incident — often contains the label outright. |

**Unused** — identifiers, constants, or redundant fields with no modeling value.

In [ ]:
# Proof: Distance(mi) is leakage — it scales directly with severity
print("Distance(mi) by Severity (proof of leakage):")
print(df.groupby("Severity")["Distance(mi)"].agg(["mean", "median", "max"]).round(2))

In [ ]:
LEAKAGE_COLS = ["End_Time", "End_Lat", "End_Lng", "Distance(mi)", "Description"]
UNUSED_COLS = [
    "ID", "Source", "Country", "Turning_Loop",      # constants / identifiers
    "Weather_Timestamp", "Airport_Code",             # ancillary metadata
    "Street", "City", "County", "Zipcode", "Timezone",  # high-cardinality / redundant with State
]

cols_to_drop = [c for c in LEAKAGE_COLS + UNUSED_COLS if c in df.columns]
print(f"Dropping {len(LEAKAGE_COLS)} leakage cols + "
      f"{len(UNUSED_COLS)} unused cols = {len(cols_to_drop)} total")
df = df.drop(columns=cols_to_drop)
print(f"Shape: {df.shape}")

### 4.2 Missing-value audit

In [ ]:
missing = df.isna().sum().sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({"missing": missing, "pct": missing_pct})
print("Columns with missing values:")
missing_df[missing_df["missing"] > 0]

In [ ]:
# Visualize missingness
to_plot = missing_df[missing_df["pct"] > 0].sort_values("pct", ascending=True)

fig, ax = plt.subplots(figsize=(8, max(3, 0.3 * len(to_plot))))
ax.barh(to_plot.index, to_plot["pct"],
        color=["indianred" if p > 50 else "steelblue" for p in to_plot["pct"]])
ax.axvline(50, color="black", ls="--", label="50% threshold (drop)")
ax.set_xlabel("% missing")
ax.set_title("Missingness by column")
ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "00_missingness.png", dpi=120)
plt.show()

### 4.3 Drop high-missingness features (>50%)

Two columns clear the 50% threshold and aren't worth imputing — the signal-to-noise on imputed values would be poor.

In [ ]:
HIGH_MISSING_THRESHOLD = 0.50
high_missing = (df.isna().sum() / len(df))
to_drop_missing = high_missing[high_missing > HIGH_MISSING_THRESHOLD].index.tolist()

print(f"Columns >50% missing → dropping: {to_drop_missing}")
df = df.drop(columns=to_drop_missing)
print(f"Shape: {df.shape}")

### 4.4 Outlier detection & handling

**Strategy:** Clip rather than drop. Reasons:
1. We have 7M rows — even if 1% are outliers, dropping discards 70K real accident records.
2. Many "outliers" in weather features (e.g., -30°F in Minnesota in January, 50 mph wind in a hurricane) are *real* and informative.
3. Tree models (our chosen family) are robust to outliers, but extreme tails still slow training and skew StandardScaler.

**Method:** IQR rule with a wider bound (3.0× instead of the standard 1.5×) — we only clip *truly extreme* values, not normal seasonal variation. Then clip at the resulting bounds rather than drop the row.

In [ ]:
NUMERIC_OUTLIER_COLS = [
    "Temperature(F)", "Humidity(%)", "Pressure(in)",
    "Visibility(mi)", "Wind_Speed(mph)",
]

def iqr_bounds(s, k=3.0):
    """Returns (lower, upper) bounds via the IQR rule. k=3.0 = far-out outliers only."""
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    return q1 - k * iqr, q3 + k * iqr

outlier_report = []
for col in NUMERIC_OUTLIER_COLS:
    if col not in df.columns:
        continue
    s = df[col].dropna()
    lo, hi = iqr_bounds(s)
    n_low = (s < lo).sum()
    n_high = (s > hi).sum()
    outlier_report.append({
        "feature": col,
        "q1": s.quantile(0.25), "q3": s.quantile(0.75),
        "lower_bound": lo, "upper_bound": hi,
        "n_below": n_low, "n_above": n_high,
        "pct_outliers": (n_low + n_high) / len(s) * 100,
    })

outlier_df = pd.DataFrame(outlier_report).round(2)
print("Outlier report (IQR rule, k=3.0):")
outlier_df

In [ ]:
# Visualize before clipping
fig, axes = plt.subplots(1, len(NUMERIC_OUTLIER_COLS),
                         figsize=(3 * len(NUMERIC_OUTLIER_COLS), 4))
for ax, col in zip(axes, NUMERIC_OUTLIER_COLS):
    if col in df.columns:
        df[col].plot.box(ax=ax, vert=True)
        ax.set_title(col, fontsize=10)
        ax.set_xticks([])
plt.suptitle("Numeric features — boxplots BEFORE clipping", y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR / "00_outliers_before.png", dpi=120)
plt.show()

In [ ]:
# Clip
print("Clipping outliers (winsorization):")
for col in NUMERIC_OUTLIER_COLS:
    if col not in df.columns:
        continue
    lo, hi = iqr_bounds(df[col].dropna())
    before_min, before_max = df[col].min(), df[col].max()
    df[col] = df[col].clip(lower=lo, upper=hi)
    print(f"  {col:20s}  was [{before_min:.1f}, {before_max:.1f}] "
          f"→ now [{df[col].min():.1f}, {df[col].max():.1f}]")

In [ ]:
# Visualize after clipping
fig, axes = plt.subplots(1, len(NUMERIC_OUTLIER_COLS),
                         figsize=(3 * len(NUMERIC_OUTLIER_COLS), 4))
for ax, col in zip(axes, NUMERIC_OUTLIER_COLS):
    if col in df.columns:
        df[col].plot.box(ax=ax, vert=True)
        ax.set_title(col, fontsize=10)
        ax.set_xticks([])
plt.suptitle("Numeric features — boxplots AFTER clipping", y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR / "00_outliers_after.png", dpi=120)
plt.show()

### 4.5 Impute missing values

**Numerics → median** (robust to skewed distributions, unlike the mean).
**Categoricals → "Unknown" as an explicit category** (lets the model learn that "missing" is itself signal — e.g., poorly-monitored road segments).

Booleans (POI flags) come through as `False`/`True` with no missing entries — left as-is for now (we'll cast in 4.6).

In [ ]:
# Identify column types from what's left
remaining_num = df.select_dtypes(include=["number"]).columns.tolist()
remaining_cat = df.select_dtypes(include=["object", "string"]).columns.tolist()
remaining_bool = df.select_dtypes(include=["bool"]).columns.tolist()

print(f"Numeric:     {remaining_num}")
print(f"Categorical: {remaining_cat}")
print(f"Boolean:     {remaining_bool}")

In [ ]:
# Impute
missing_before = df.isna().sum().sum()

for c in remaining_num:
    if df[c].isna().any():
        df[c] = df[c].fillna(df[c].median())
for c in remaining_cat:
    if df[c].isna().any():
        df[c] = df[c].fillna("Unknown")

missing_after = df.isna().sum().sum()
print(f"Total missing cells: {missing_before:,} → {missing_after:,}")

### 4.6 Type casting — downcast to save memory

**Before:** Defaults are `float64` for numerics and `object` for strings. On 7M rows this is wasteful — `float64` uses 8 bytes per value where `float32` only needs 4, and `category` dtype stores each unique string once.

**After:** Roughly halves the in-memory footprint, which speeds up every downstream step.

In [ ]:
print(f"Memory BEFORE casting: {df.memory_usage(deep=True).sum() / 1e9:.2f} GB\n")

# Booleans: True/False → int8 (LightGBM and most models prefer numeric)
for c in remaining_bool:
    df[c] = df[c].astype("int8")

# Numerics: downcast to smallest safe float / int
for c in remaining_num:
    if df[c].dtype.kind == "f":          # floats
        df[c] = pd.to_numeric(df[c], downcast="float")
    elif df[c].dtype.kind in "iu":       # ints
        df[c] = pd.to_numeric(df[c], downcast="integer")

# Categoricals: convert to pandas `category` dtype
for c in remaining_cat:
    df[c] = df[c].astype("category")

print(f"Memory AFTER casting:  {df.memory_usage(deep=True).sum() / 1e9:.2f} GB")
print("\nFinal dtypes:")
print(df.dtypes.value_counts())

### 4.7 Variance check (sanity)

Quick sanity check that we haven't kept any zero-variance numeric columns (constants are useless to the model). For categoricals we already dropped the obvious constants (`Country`, `Turning_Loop`) in Section 4.1.

In [ ]:
numeric_now = df.select_dtypes(include=["number"]).columns
variances = df[numeric_now].var().sort_values()
print("Lowest-variance numeric features:")
variances.head(10)

All numerics have meaningful variance. No further drops needed.

**Data preparation complete.** Summary of what we did:

| Step | Action | Result |
|---|---|---|
| 4.1 | Drop leakage + unused | -16 cols |
| 4.3 | Drop >50% missing | -2 cols (`Precipitation(in)`, `Wind_Chill(F)`) |
| 4.4 | Clip outliers (IQR, k=3) | Numerics bounded |
| 4.5 | Impute (median / "Unknown") | 0 missing cells |
| 4.6 | Downcast types | ~50% memory reduction |
| 4.7 | Variance check | Confirmed no constants |

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# FEATURE ENGINEERING
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## 5. Feature engineering

### 5.1 Time features

In [ ]:
df["Start_Time"] = pd.to_datetime(df["Start_Time"], errors="coerce", format="mixed")
df = df.dropna(subset=["Start_Time"])

df["Hour"] = df["Start_Time"].dt.hour.astype("int8")
df["DayOfWeek"] = df["Start_Time"].dt.dayofweek.astype("int8")
df["Month"] = df["Start_Time"].dt.month.astype("int8")
df["Year"] = df["Start_Time"].dt.year.astype("int16")
df["IsWeekend"] = (df["DayOfWeek"] >= 5).astype("int8")
df["IsRushHour"] = (
    df["Hour"].isin([7, 8, 9, 16, 17, 18]) & (df["DayOfWeek"] < 5)
).astype("int8")

df[["Start_Time", "Hour", "DayOfWeek", "Month", "Year",
    "IsWeekend", "IsRushHour"]].head()

### 5.2 Weather buckets

`Weather_Condition` has 100+ unique values. Most appear in <0.1% of rows — we collapse them into ~10 meaningful buckets so the model learns stable categories instead of overfitting rare strings.

In [ ]:
print(f"Unique Weather_Condition values: {df['Weather_Condition'].nunique()}")
df["Weather_Condition"].value_counts().head(10)

In [ ]:
WEATHER_MAP = {
    "Clear": "Clear", "Fair": "Clear", "Mostly Clear": "Clear",
    "Cloudy": "Cloudy", "Mostly Cloudy": "Cloudy", "Partly Cloudy": "Cloudy",
    "Overcast": "Cloudy", "Scattered Clouds": "Cloudy",
    "Rain": "Rain", "Light Rain": "Rain", "Heavy Rain": "Rain",
    "Drizzle": "Rain", "Light Drizzle": "Rain", "Showers in the Vicinity": "Rain",
    "Snow": "Snow", "Light Snow": "Snow", "Heavy Snow": "Snow", "Blowing Snow": "Snow",
    "Fog": "Fog", "Haze": "Fog", "Mist": "Fog", "Patches of Fog": "Fog",
    "Thunderstorm": "Storm", "T-Storm": "Storm", "Thunder": "Storm",
    "Wintry Mix": "Snow", "Sleet": "Snow", "Ice Pellets": "Snow",
}
df["Weather_Group"] = (
    df["Weather_Condition"]
    .astype(str).map(WEATHER_MAP).fillna("Other").astype("category")
)
df["Weather_Group"].value_counts()

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# EXPLORATORY DATA ANALYSIS
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## 6. Exploratory Data Analysis

### 6.1 Target distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

sev_counts = df["Severity"].value_counts().sort_index()
axes[0].bar(sev_counts.index.astype(str), sev_counts.values, color="steelblue")
axes[0].set_title("4-class Severity distribution")
axes[0].set_xlabel("Severity (1=light → 4=major)")
axes[0].set_ylabel("Accidents")
for i, v in enumerate(sev_counts.values):
    axes[0].text(i, v, f"{v/sev_counts.sum():.1%}", ha="center", va="bottom")

bin_target = (df["Severity"] >= 3).astype(int)
bin_counts = bin_target.value_counts().sort_index()
axes[1].bar(["Mild (1-2)", "Severe (3-4)"], bin_counts.values,
            color=["seagreen", "indianred"])
axes[1].set_title("Binarized target (for modeling)")
axes[1].set_ylabel("Accidents")
for i, v in enumerate(bin_counts.values):
    axes[1].text(i, v, f"{v/bin_counts.sum():.1%}", ha="center", va="bottom")

plt.tight_layout()
plt.savefig(FIG_DIR / "01_target_distribution.png", dpi=120)
plt.show()

**Insight:** Severity 2 dominates (~80%). After binarizing, ~20% are severe. Accuracy is useless as a metric — predicting "mild" every time hits 80%. We'll use **macro-F1 and PR-AUC**.

### 6.2 Accidents over time

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

hour_data = df.groupby("Hour").agg(
    total=("Severity", "size"),
    severe_rate=("Severity", lambda x: (x >= 3).mean()),
)
axes[0, 0].bar(hour_data.index, hour_data["total"], color="steelblue", alpha=0.8)
axes[0, 0].set_title("Accidents by hour of day")
axes[0, 0].set_xlabel("Hour"); axes[0, 0].set_ylabel("Accidents")

axes[0, 1].plot(hour_data.index, hour_data["severe_rate"], color="indianred", marker="o")
axes[0, 1].axhline(hour_data["severe_rate"].mean(), color="black", ls="--",
                   label=f"Mean = {hour_data['severe_rate'].mean():.2f}")
axes[0, 1].set_title("P(severe) by hour of day")
axes[0, 1].set_xlabel("Hour"); axes[0, 1].set_ylabel("P(severe)"); axes[0, 1].legend()

dow_data = df.groupby("DayOfWeek").size()
day_names = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
axes[1, 0].bar(day_names, dow_data.values, color="steelblue", alpha=0.8)
axes[1, 0].set_title("Accidents by day of week"); axes[1, 0].set_ylabel("Accidents")

month_data = df.groupby("Month").size()
axes[1, 1].bar(month_data.index, month_data.values, color="steelblue", alpha=0.8)
axes[1, 1].set_title("Accidents by month")
axes[1, 1].set_xlabel("Month"); axes[1, 1].set_ylabel("Accidents")

plt.tight_layout()
plt.savefig(FIG_DIR / "02_time_patterns.png", dpi=120)
plt.show()

**Insight:** Volume peaks at rush hours (7–9am, 4–6pm). The *severe-rate* also climbs at those times. Weekdays dominate over weekends by volume.

### 6.3 Yearly trend

In [ ]:
yearly = df.groupby("Year").size()
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(yearly.index, yearly.values, marker="o", linewidth=2, color="steelblue")
ax.set_title("Accidents per year")
ax.set_xlabel("Year"); ax.set_ylabel("Accidents")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / "03_yearly_trend.png", dpi=120)
plt.show()

**Insight:** Sharp drop in 2020 reflects COVID lockdowns. Steady growth otherwise — partly real, partly data-collection coverage expanding.

### 6.4 Geographic distribution

In [ ]:
state_stats = df.groupby("State", observed=True).agg(
    total=("Severity", "size"),
    severe_rate=("Severity", lambda x: (x >= 3).mean()),
).sort_values("total", ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

top15 = state_stats.head(15)
axes[0].barh(top15.index[::-1], top15["total"][::-1], color="steelblue")
axes[0].set_title("Top 15 states by accident volume")
axes[0].set_xlabel("Accidents")

sev_top = state_stats[state_stats["total"] > 5000].sort_values(
    "severe_rate", ascending=False
).head(15)
axes[1].barh(sev_top.index[::-1], sev_top["severe_rate"][::-1], color="indianred")
axes[1].axvline(df["Severity"].ge(3).mean(), color="black", ls="--",
                label=f"National mean = {df['Severity'].ge(3).mean():.2f}")
axes[1].set_title("Top 15 states by severe-rate (≥5k accidents)")
axes[1].set_xlabel("P(severe)"); axes[1].legend()

plt.tight_layout()
plt.savefig(FIG_DIR / "04_geographic.png", dpi=120)
plt.show()

**Insight:** California, Florida, Texas dominate by volume — partly population, partly data-collection coverage. Severe-rate varies meaningfully across states, so `State` will be a useful feature.

### 6.5 Lat/lng scatter

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))
sample = df.sample(min(100_000, len(df)), random_state=42)
sc = ax.scatter(
    sample["Start_Lng"], sample["Start_Lat"],
    c=sample["Severity"], cmap="YlOrRd",
    s=1, alpha=0.3,
)
ax.set_title("Accident locations (100k sample) — color = severity")
ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")
ax.set_xlim(-130, -65); ax.set_ylim(23, 50)
plt.colorbar(sc, ax=ax, label="Severity")
plt.tight_layout()
plt.savefig(FIG_DIR / "05_lat_lng_scatter.png", dpi=120)
plt.show()

**Insight:** The scatter traces the US Interstate network — accidents cluster along major freeways and metro areas.

### 6.6 Weather

In [ ]:
weather_stats = df.groupby("Weather_Group", observed=True).agg(
    total=("Severity", "size"),
    severe_rate=("Severity", lambda x: (x >= 3).mean()),
).sort_values("total", ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].barh(weather_stats.index[::-1], weather_stats["total"][::-1], color="steelblue")
axes[0].set_title("Accident volume by weather"); axes[0].set_xlabel("Accidents")

axes[1].barh(weather_stats.index[::-1], weather_stats["severe_rate"][::-1], color="indianred")
axes[1].axvline(df["Severity"].ge(3).mean(), color="black", ls="--", label="Overall mean")
axes[1].set_title("Severe-rate by weather"); axes[1].set_xlabel("P(severe)"); axes[1].legend()

plt.tight_layout()
plt.savefig(FIG_DIR / "06_weather.png", dpi=120)
plt.show()
weather_stats.round(3)

**Insight:** Most accidents happen in clear weather (because most driving does). Storms, fog, and snow have higher per-trip risk but contribute fewer absolute accidents.

### 6.7 Road feature lift (POI flags)

In [ ]:
BOOL_FEATURES = ["Amenity", "Bump", "Crossing", "Give_Way", "Junction",
                 "No_Exit", "Railway", "Roundabout", "Station", "Stop",
                 "Traffic_Calming", "Traffic_Signal"]

bool_stats = pd.DataFrame({
    feat: {
        "prevalence": df[feat].mean(),
        "p_severe|true": df.loc[df[feat] == 1, "Severity"].ge(3).mean(),
        "p_severe|false": df.loc[df[feat] == 0, "Severity"].ge(3).mean(),
    }
    for feat in BOOL_FEATURES if feat in df.columns
}).T
bool_stats["lift"] = (
    bool_stats["p_severe|true"] / bool_stats["p_severe|false"]
).round(2)
bool_stats = bool_stats.sort_values("lift", ascending=False)

fig, ax = plt.subplots(figsize=(8, 5))
colors = ["indianred" if v > 1 else "seagreen" for v in bool_stats["lift"]]
ax.barh(bool_stats.index[::-1], bool_stats["lift"][::-1], color=colors[::-1])
ax.axvline(1.0, color="black", ls="--", label="No effect")
ax.set_xlabel("Lift = P(severe | feature) / P(severe | no feature)")
ax.set_title("Road-feature lift on severe-rate")
ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "07_road_features.png", dpi=120)
plt.show()

bool_stats.round(3)

**Insight:** Traffic_Signal, Crossing, and Junction have meaningful *negative* lift — signaled intersections cap speeds, so accidents there are less severe in flow-impact terms.

### 6.8 Numeric distributions by severity (post-clipping)

In [ ]:
NUMERIC_FEATURES = ["Temperature(F)", "Humidity(%)", "Pressure(in)",
                    "Visibility(mi)", "Wind_Speed(mph)"]

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
axes = axes.flatten()

for i, col in enumerate(NUMERIC_FEATURES):
    sample = df[[col, "Severity"]].dropna().sample(
        min(100_000, len(df)), random_state=42
    )
    sample["sev_binary"] = np.where(sample["Severity"] >= 3, "Severe", "Mild")
    sns.violinplot(
        data=sample, x="sev_binary", y=col, ax=axes[i],
        palette={"Mild": "seagreen", "Severe": "indianred"},
        inner="quartile",
    )
    axes[i].set_title(col); axes[i].set_xlabel("")

axes[-1].axis("off")
plt.tight_layout()
plt.savefig(FIG_DIR / "08_numeric_by_severity.png", dpi=120)
plt.show()

**Insight:** Distributions look nearly identical mild vs severe — no single numeric feature separates the classes well. Section 6.10 confirms this statistically.

### 6.9 Correlation heatmap

In [ ]:
df["severe_binary"] = (df["Severity"] >= 3).astype(int)
corr_cols = NUMERIC_FEATURES + ["Hour", "DayOfWeek", "Month",
                                "IsWeekend", "IsRushHour", "severe_binary"]
corr = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
            square=True, cbar_kws={"shrink": 0.8}, ax=ax)
ax.set_title("Correlation: numeric features and severe target")
plt.tight_layout()
plt.savefig(FIG_DIR / "09_correlation.png", dpi=120)
plt.show()

**Insight:** No correlation above ~0.05 with the target. Severity is driven by **interactions** of categorical + numeric features, not by any single variable. Tree-based models will capture this.

### 6.10 Statistical tests — are these differences real?

**Tests used:**
- **Mann–Whitney U** for each numeric: does its distribution differ between severe and mild? (Non-parametric, robust to non-normality.)
- **Chi-square + Cramér's V** for categoricals: is the feature associated with severity? (Cramér's V gives effect size, which matters because chi-square p-values are misleadingly tiny on large samples.)

#### Mann–Whitney U on numeric features

In [ ]:
severe = df[df["severe_binary"] == 1]
mild = df[df["severe_binary"] == 0]

sev_sample = severe.sample(min(50_000, len(severe)), random_state=42)
mild_sample = mild.sample(min(50_000, len(mild)), random_state=42)

mwu_results = []
for col in NUMERIC_FEATURES:
    a = sev_sample[col].dropna()
    b = mild_sample[col].dropna()
    stat, p = stats.mannwhitneyu(a, b, alternative="two-sided")
    n1, n2 = len(a), len(b)
    mean_u = n1 * n2 / 2
    std_u = np.sqrt(n1 * n2 * (n1 + n2 + 1) / 12)
    z = (stat - mean_u) / std_u
    r = abs(z) / np.sqrt(n1 + n2)
    mwu_results.append({
        "feature": col,
        "median_severe": a.median(),
        "median_mild": b.median(),
        "p_value": p,
        "effect_size_r": r,
    })

mwu_df = pd.DataFrame(mwu_results).sort_values("effect_size_r", ascending=False)
mwu_df.round({"median_severe": 2, "median_mild": 2,
              "p_value": 6, "effect_size_r": 4})

**Reading this:** With 100k observations almost every difference is "statistically significant" (p < 0.05), but effect sizes (`r`) are tiny — all below 0.1. Rule of thumb: r < 0.1 is negligible, 0.1–0.3 small, 0.3–0.5 medium, >0.5 large.

**Conclusion:** No individual numeric feature has a meaningful effect.

#### Chi-square + Cramér's V on categoricals

In [ ]:
def cramers_v(confusion_matrix):
    chi2 = stats.chi2_contingency(confusion_matrix)[0]
    n = confusion_matrix.sum().sum()
    r, k = confusion_matrix.shape
    return np.sqrt(chi2 / (n * (min(r, k) - 1)))

CATEGORICAL_FEATURES = ["State", "Weather_Group", "Sunrise_Sunset", "Wind_Direction"]
CATEGORICAL_FEATURES = [c for c in CATEGORICAL_FEATURES if c in df.columns]

cat_results = []
for col in CATEGORICAL_FEATURES:
    ctab = pd.crosstab(df[col], df["severe_binary"])
    chi2, p, dof, _ = stats.chi2_contingency(ctab)
    v = cramers_v(ctab)
    cat_results.append({
        "feature": col,
        "n_categories": ctab.shape[0],
        "chi2": chi2,
        "p_value": p,
        "cramers_v": v,
    })

cat_df = pd.DataFrame(cat_results).sort_values("cramers_v", ascending=False)
cat_df.round({"chi2": 1, "p_value": 6, "cramers_v": 4})

**Conclusion:** State and Weather_Group show the strongest associations with severity. Sunrise_Sunset and Wind_Direction are weak.

#### Combined view

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].barh(mwu_df["feature"][::-1], mwu_df["effect_size_r"][::-1], color="steelblue")
axes[0].set_title("Numeric features — Mann-Whitney U effect size (r)")
axes[0].set_xlabel("Effect size r")
axes[0].axvline(0.1, color="black", ls="--", label="r = 0.1 (small)")
axes[0].legend()

axes[1].barh(cat_df["feature"][::-1], cat_df["cramers_v"][::-1], color="indianred")
axes[1].set_title("Categorical features — Cramér's V")
axes[1].set_xlabel("Cramér's V")
axes[1].axvline(0.1, color="black", ls="--", label="V = 0.1 (small)")
axes[1].legend()

plt.tight_layout()
plt.savefig(FIG_DIR / "10_statistical_tests.png", dpi=120)
plt.show()

**Bottom line:** Categorical features (especially State and Weather_Group) do much more work than individual numerics. This justifies tree-based models that handle categorical interactions natively.

## 7. Summary of EDA insights

1. **Severity is heavily imbalanced** — ~80% class 2. Use macro-F1 / PR-AUC, not accuracy.
2. **Time of day matters** — volume and severe-rate both peak at rush hours.
3. **Geography matters most** — State has the largest Cramér's V (strongest association with the target).
4. **Weather is bucketed** — clear-weather accidents dominate by volume; storms/fog/snow have higher per-trip risk.
5. **Road features show meaningful lift** — Traffic_Signal, Junction, Crossing all reduce severe-rate.
6. **No single numeric variable separates classes well** (effect sizes < 0.1) → tree-based models needed.
7. **Distance(mi) was dropped as target leakage** — see Section 4.1.

## 8. Save cleaned data for the modeling notebook

In [ ]:
df.to_parquet(CLEAN_PARQUET, compression="snappy", index=False)
print(f"Saved cleaned data → {CLEAN_PARQUET} "
      f"({CLEAN_PARQUET.stat().st_size / 1e6:.0f} MB)")
print(f"Final shape: {df.shape}")
print(f"\nReady for notebooks/02_modeling.ipynb")